In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("load_date", "")
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("source_system", "retail_pos")
dbutils.widgets.text("catalog_name", "retail_lakehouse")

environment = dbutils.widgets.get("environment")
load_date = dbutils.widgets.get("load_date")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
source_system = dbutils.widgets.get("source_system")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
from datetime import date
import uuid

if not load_date:
    load_date = date.today().strftime("%Y-%m-%d")

if not pipeline_run_id:
    pipeline_run_id = str(uuid.uuid4())

print(f"Environment     : {environment}")
print(f"Load date       : {load_date}")
print(f"Pipeline run ID : {pipeline_run_id}")
print(f"Source system   : {source_system}")
print(f"Catalog         : {catalog_name}")

In [0]:
source_path = (
    f"/Volumes/{catalog_name}/landing/source_files/"
    f"sales/load_date={load_date}"
)

bronze_table = (
    f"{catalog_name}.bronze.sales_transactions"
)

quarantine_table = (
    f"{catalog_name}.quarantine.sales_malformed_records"
)

audit_table = (
    f"{catalog_name}.audit.pipeline_runs"
)

processed_files_table = (
    f"{catalog_name}.audit.processed_files"
)

print(f"Source path    : {source_path}")
print(f"Bronze table   : {bronze_table}")
print(f"Quarantine     : {quarantine_table}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {processed_files_table}
(
    source_file_path STRING,
    source_file_name STRING,
    source_system STRING,
    target_table STRING,
    file_size_bytes BIGINT,
    file_modification_time TIMESTAMP,
    pipeline_run_id STRING,
    processed_timestamp TIMESTAMP,
    processing_status STRING
)
USING DELTA
COMMENT 'Tracks source files processed by the retail lakehouse'
""")

In [0]:
spark.sql(
    f"DESCRIBE TABLE {processed_files_table}"
).show(truncate=False)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

sales_raw_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("transaction_timestamp", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount_amount", StringType(), True),
    StructField("status_code", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("sales_channel", StringType(), True),
    StructField("promotion_id", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
source_files = [
    file
    for file in dbutils.fs.ls(source_path)
    if file.name.lower().endswith(".csv")
]

if not source_files:
    raise FileNotFoundError(
        f"No CSV files found at {source_path}"
    )

for file in source_files:
    print(
        file.name,
        file.size,
        file.modificationTime
    )

In [0]:
available_file_paths = [
    file.path
    for file in source_files
]

available_file_paths

In [0]:
from pyspark.sql.functions import col

processed_file_rows = (
    spark.table(processed_files_table)
    .filter(
        (col("target_table") == bronze_table)
        & (col("processing_status") == "SUCCESS")
    )
    .select("source_file_path")
    .distinct()
    .collect()
)

processed_file_paths = {
    row["source_file_path"]
    for row in processed_file_rows
}

new_file_paths = [
    path
    for path in available_file_paths
    if path not in processed_file_paths
]

print(f"Available files : {len(available_file_paths)}")
print(f"Processed files : {len(processed_file_paths)}")
print(f"New files       : {len(new_file_paths)}")

In [0]:
if not new_file_paths:
    dbutils.notebook.exit(
        f"No new sales files found for load_date={load_date}"
    )

In [0]:
from datetime import datetime

pipeline_name = "retail_sales_ingestion"
notebook_name = "01_ingest_sales_bronze"
start_timestamp = datetime.now()

records_read = 0
records_written = 0
records_rejected = 0
run_status = "RUNNING"
error_message = None

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

def write_pipeline_audit(
    run_status,
    records_read=0,
    records_written=0,
    records_rejected=0,
    error_message=None
):
    audit_record = [
        Row(
            pipeline_run_id=pipeline_run_id,
            pipeline_name=pipeline_name,
            notebook_name=notebook_name,
            source_system=source_system,
            source_file_name=",".join(
                [path.split("/")[-1] for path in new_file_paths]
            ),
            source_file_path=",".join(new_file_paths),
            layer="bronze",
            load_type="INCREMENTAL",
            run_status=run_status,
            records_read=records_read,
            records_written=records_written,
            records_rejected=records_rejected,
            start_timestamp=start_timestamp,
            end_timestamp=datetime.now(),
            error_message=error_message,
            created_timestamp=datetime.now()
        )
    ]

    (
        spark.createDataFrame(audit_record)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(audit_table)
    )

In [0]:
from pyspark.sql import functions as F

raw_sales_df = (
    spark.read
    .format("csv")
    .schema(sales_raw_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option(
        "columnNameOfCorruptRecord",
        "_corrupt_record"
    )
    .option("quote", '"')
    .option("escape", '"')
    .load(new_file_paths)
)

In [0]:
from pyspark.sql import functions as F

raw_sales_df = (
    spark.read
    .format("csv")
    .schema(sales_raw_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option(
        "columnNameOfCorruptRecord",
        "_corrupt_record"
    )
    .option("quote", '"')
    .option("escape", '"')
    .load(new_file_paths)
)

In [0]:
raw_sales_df.printSchema()

In [0]:
display(raw_sales_df.limit(20))

In [0]:
records_read = raw_sales_df.count()

print(f"Records read: {records_read}")

In [0]:
bronze_sales_df = (
    raw_sales_df
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_file_name",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"([^/]+$)",
            1
        )
    )
    .withColumn(
        "_source_file_size",
        F.col("_metadata.file_size")
    )
    .withColumn(
        "_source_file_modification_time",
        F.col("_metadata.file_modification_time")
    )
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_ingestion_date",
        F.current_date()
    )
    .withColumn(
        "_source_load_date",
        F.to_date(F.lit(load_date))
    )
    .withColumn(
        "_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_source_system",
        F.lit(source_system)
    )
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(
                    F.col("transaction_id"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("customer_id"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("product_id"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("store_id"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("transaction_timestamp"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("quantity"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("unit_price"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("discount_amount"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("status_code"),
                    F.lit("")
                )
            ),
            256
        )
    )
)

In [0]:
display(
    bronze_sales_df.select(
        "transaction_id",
        "quantity",
        "transaction_timestamp",
        "_source_file_name",
        "_ingestion_timestamp",
        "_pipeline_run_id",
        "_record_hash",
        "_corrupt_record"
    ).limit(20)
)

In [0]:
malformed_df = bronze_sales_df.filter(
    F.col("_corrupt_record").isNotNull()
)

valid_structure_df = bronze_sales_df.filter(
    F.col("_corrupt_record").isNull()
)

records_rejected = malformed_df.count()
records_written = valid_structure_df.count()

print(f"Structurally valid records : {records_written}")
print(f"Malformed CSV records      : {records_rejected}")

In [0]:
(
    valid_structure_df
    .write
    .format("delta")
    .mode("append")
    .partitionBy("_ingestion_date")
    .saveAsTable(bronze_table)
)

print(
    f"{records_written} rows written to {bronze_table}"
)

In [0]:
from datetime import datetime

file_metadata_by_path = {
    file.path: file
    for file in source_files
}

processed_file_records = []

for file_path in new_file_paths:
    file_info = file_metadata_by_path[file_path]

    processed_file_records.append(
        Row(
            source_file_path=file_path,
            source_file_name=file_info.name,
            source_system=source_system,
            target_table=bronze_table,
            file_size_bytes=file_info.size,
            file_modification_time=datetime.fromtimestamp(
                file_info.modificationTime / 1000
            ),
            pipeline_run_id=pipeline_run_id,
            processed_timestamp=datetime.now(),
            processing_status="SUCCESS"
        )
    )

(
    spark.createDataFrame(processed_file_records)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(processed_files_table)
)

In [0]:
write_pipeline_audit(
    run_status="SUCCESS",
    records_read=records_read,
    records_written=records_written,
    records_rejected=records_rejected,
    error_message=""
)

In [0]:
result = {
    "pipeline_run_id": pipeline_run_id,
    "status": "SUCCESS",
    "records_read": records_read,
    "records_written": records_written,
    "records_rejected": records_rejected,
    "target_table": bronze_table
}

print(result)

In [0]:
spark.table(bronze_table).printSchema()

In [0]:
display(
    spark.table(bronze_table)
    .select(
        "transaction_id",
        "customer_id",
        "product_id",
        "quantity",
        "transaction_timestamp",
        "status_code",
        "_source_file_name",
        "_source_load_date",
        "_ingestion_timestamp"
    )
    .limit(20)
)

In [0]:
bronze_count = spark.table(bronze_table).count()

print(f"Total Bronze records: {bronze_count}")

In [0]:
display(
    spark.table(bronze_table)
    .filter(
        (F.col("quantity").isin("", "-1", "ABC"))
        | (F.col("quantity").isNull())
        | (F.col("transaction_timestamp") == "INVALID_DATE")
        | (F.col("product_id") == "P9999")
        | (F.col("status_code").isin("X", "UNKNOWN", ""))
    )
)

In [0]:
display(
    spark.table(audit_table)
    .filter(
        F.col("pipeline_run_id") == pipeline_run_id
    )
)

In [0]:
display(
    spark.table(processed_files_table)
    .orderBy(
        F.col("processed_timestamp").desc()
    )
)

In [0]:
try:
    # Read
    # Add metadata
    # Write quarantine
    # Write Bronze
    # Register processed files
    # Write success audit

    pass

except Exception as exc:
    error_message = str(exc)[:4000]

    write_pipeline_audit(
        run_status="FAILED",
        records_read=records_read,
        records_written=records_written,
        records_rejected=records_rejected,
        error_message=error_message
    )

    raise